In [87]:
import pandas as pd

data = pd.read_csv("../inputs/rolled_player_data_23-24_all_players_xp_fixed.csv")
data = data[data["minutes_rolling_3"] > 15]


In [76]:
import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostRegressor
from sklearn import (
    compose,
    dummy,
    ensemble,
    impute,
    linear_model,
    preprocessing,
    svm,
    tree,
    metrics,
)
from sklearn.pipeline import make_pipeline

In [ ]:
num_cols = [
    "elo_h",
    "elo_a",
    # "opp_elo",
    # "opp_elo_h",
    # "opp_elo_a",
    # "team_h_difficulty",
    # "team_a_difficulty",
    "win_prob",
    "value",
    "selected_by_percent",
    "transfers_in",
    "transfers_out",
    "total_points_rolling_3",
    "xGChain_rolling_3",
    "xGBuildup_rolling_3",
    "expected_goals_rolling_3",
    "expected_assists_rolling_3",
    "expected_goal_involvements_rolling_3",
    "minutes_rolling_3",
    "starts_rolling_3",
    "expected_goals_conceded_rolling_3",
    "clean_sheets_rolling_3",
    # "saves_rolling_3",
    "ict_index_rolling_3",
    "influence_rolling_3",
    "creativity_rolling_3",
    "threat_rolling_3",
]

cat_cols = [
    "team",
    "opponent_team",
    "was_home",
]
# data[num_cols_gk]

In [66]:
def linear_prep(num_cols, cat_cols):
    return compose.ColumnTransformer(
        [
            (
                "num",
                make_pipeline(
                    impute.SimpleImputer(strategy="median"),
                    preprocessing.StandardScaler(),
                ),
                num_cols,
            ),
            ("cat", preprocessing.OneHotEncoder(handle_unknown="ignore"), cat_cols),
        ]
    )


def features(df):
    """Model inputs: every column except the identifiers, the target and `row`."""
    return df.drop(columns=DROP_COLS + ["row"], errors="ignore")

In [96]:
pos_data = data[data["position"] == "DEF"]
df_train = pos_data[pos_data["round"] < 20].reset_index(drop=True)
df_valid = pos_data[pos_data["round"] == 21].reset_index(drop=True)
DROP_COLS = ["fpl_id", "round", "total_points"]

x_train = df_train[num_cols + cat_cols]  # .drop(columns=DROP_COLS)
y_train = df_train.total_points

x_valid = df_valid[num_cols + cat_cols]  # .drop(columns=DROP_COLS)
y_valid = df_valid.total_points

y_train

0       5
1       1
2       6
3       8
4       1
       ..
1696    0
1697    0
1698    0
1699    1
1700    1
Name: total_points, Length: 1701, dtype: int64

In [92]:
## Models
def make_models(num_cols):
    """Fresh, unfitted models for one position's numeric feature list."""
    return {
        "dummy": dummy.DummyRegressor(strategy="median"),  # floor
        "ridge": make_pipeline(
            linear_prep(num_cols, cat_cols), linear_model.Ridge(alpha=1.0)
        ),
        "dt": make_pipeline(
            tree_prep(num_cols, cat_cols),
            tree.DecisionTreeRegressor(
                max_depth=6, min_samples_leaf=30, random_state=42
            ),
        ),
        "rf": make_pipeline(
            tree_prep(num_cols, cat_cols),
            ensemble.RandomForestRegressor(
                n_estimators=300, min_samples_leaf=10, n_jobs=-1, random_state=42
            ),
        ),
        "hgb": make_pipeline(
            tree_prep(num_cols, cat_cols),
            ensemble.HistGradientBoostingRegressor(random_state=42),
        ),
        "lgbm": make_pipeline(
            tree_prep(num_cols, cat_cols),
            lgb.LGBMRegressor(
                n_estimators=300,
                learning_rate=0.03,
                num_leaves=15,
                min_child_samples=30,
                subsample=0.8,
                subsample_freq=1,
                colsample_bytree=0.8,
                random_state=42,
                verbose=-1,
            ),
        ),
        "xgb": make_pipeline(
            tree_prep(num_cols, cat_cols),
            xgb.XGBRegressor(
                n_estimators=300,
                learning_rate=0.03,
                max_depth=3,
                subsample=0.8,
                colsample_bytree=0.8,
                eval_metric="rmse",
                random_state=42,
            ),
        ),
        "cat": make_pipeline(
            tree_prep(num_cols, cat_cols),
            CatBoostRegressor(
                iterations=500,
                learning_rate=0.03,
                depth=4,
                random_seed=42,
                verbose=0,
                allow_writing_files=False,  # no catboost_info/ folder
            ),
        ),
        "svr": make_pipeline(linear_prep(num_cols, cat_cols), svm.SVR(C=1.0)),
    }


def linear_prep(num_cols, cat_cols):
    return compose.ColumnTransformer(
        [
            (
                "num",
                make_pipeline(
                    impute.SimpleImputer(strategy="median"),
                    preprocessing.StandardScaler(),
                ),
                num_cols,
            ),
            ("cat", preprocessing.OneHotEncoder(handle_unknown="ignore"), cat_cols),
        ]
    )


def tree_prep(num_cols, cat_cols):
    return compose.ColumnTransformer(
        [
            ("num", "passthrough", num_cols),
            (
                "cat",
                preprocessing.OrdinalEncoder(
                    handle_unknown="use_encoded_value", unknown_value=-1
                ),
                cat_cols,
            ),
        ]
    )


models = make_models(num_cols)
# models

In [97]:
for k, v in enumerate(models):
    reg = models[v]
    reg.fit(x_train, y_train)

    preds = reg.predict(x_valid)
    mae = metrics.mean_absolute_error(y_valid, preds)
    r2 = metrics.r2_score(y_valid, preds)

    print(f"{v}: mae = {mae}, R2 = {r2}")


dummy: mae = 1.9807692307692308, R2 = -0.15742522301906603
ridge: mae = 1.9261597747345052, R2 = 0.1754566198173434
dt: mae = 1.9578163733718574, R2 = 0.10577903759278329
rf: mae = 1.9306000385839848, R2 = 0.1938691308774375
hgb: mae = 1.9252263863023913, R2 = 0.12659210236006102
lgbm: mae = 1.9100076003056152, R2 = 0.16974455426011603
xgb: mae = 1.8975528139137448, R2 = 0.19516149426988072
cat: mae = 1.9004618766582158, R2 = 0.20625116461063542
svr: mae = 1.8113156936866786, R2 = 0.023280959551133962
